

# Pharmacy Staffing: Reading the Data from a CSV
### OPIM 5641 - Business Decision Modeling · Module 4.1
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5641-notebooks/blob/main/7_Nonlinear/Pharmacy_from_csv.ipynb)

*Run me top to bottom - **Runtime → Run all**. The solver installs in the first cell, and the data loads straight from GitHub - nothing to upload.*

This is the **same pharmacy problem** you solved in `Pharmacy - Complete_guided.ipynb`, with one change that matters more than it looks: **the data is no longer typed into the notebook.** It lives in a CSV on GitHub, and we read it with one line of `pandas`.

**Why bother?**

- **Hard-coded data is where mistakes hide.** Ten numbers you can eyeball. A hundred you cannot.
- **The model stops caring how much data there is.** Every loop below runs over `n_points`, which is read from the file. Add rows to the CSV and the model just works.
- **This is what the job actually looks like.** Nobody pastes their data into the model.

It's the same **soft coding** habit from brute force in M2.1 - *data in one place, model in another* - now applied to a real file.

## Setup

In [ ]:
%%capture
import sys
import os

if 'google.colab' in sys.modules:
    !pip install idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
Pharmacy from CSV - part 1: separating the data from the model

- OPEN: "same pharmacy problem, one change - the data is not in the notebook any more."
- Say why it matters: hard-coded data is where mistakes hide, and the model stops caring how many rows there are.
- Call back to soft coding in M2.1 brute force - data in one place, model in another. Same habit, real file.
- Show the raw CSV link in the browser if you want - it is just three columns: store, hours, revenue.
- Point at n_points = len(df) - THAT is the line that makes this scale. Nothing below hard-codes 10.
-->

## Read the data

In [ ]:
import pandas as pd
import numpy as np

# the data lives in the repo, so anyone can run this notebook with no setup
url = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/7_Nonlinear/data/pharmacy.csv'
df = pd.read_csv(url)

df

In [ ]:
# pull the two columns we need
# plain Python floats - numpy scalars can misbehave inside Pyomo expressions
X = [float(v) for v in df['hours']]      # pharmacy hours open
Z = [float(v) for v in df['revenue']]    # revenue ($) from each store

n_points = len(df)          # <- everything below loops over this
print('rows read from the CSV:', n_points)

**That `n_points` line is the whole point of this notebook.** Nothing below ever writes `10`. Drop more stores into the CSV, re-run, and every loop, constraint and plot adjusts by itself.

## Look at it first

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4.5))
plt.plot(X, Z, 'o')
plt.xlabel('pharmacy hours open'); plt.ylabel('revenue ($)')
plt.title('Revenue vs. hours open'); plt.grid(alpha=.3)
plt.show()

More hours, more revenue - **but look at the shape.** It doesn't obviously keep climbing at the same rate. Hold that thought; we'll come back to it with a power model at the end.

## A baseline with `scipy`

Before optimizing anything, get an answer you can check against.

In [ ]:
from scipy import stats

b_scipy, a_scipy, r_value, p_value, std_err = stats.linregress(X, Z)
print('a = %.4f' % a_scipy)
print('b = %.4f' % b_scipy)
print('R-squared = %.4f' % r_value**2)

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
Pharmacy from CSV - part 2: the same regression, now soft-coded

- scipy first so there is a reference answer to trust: a = 4435.08, b = 47.07, R-squared = 0.76.
- Then build it in Pyomo and get the SAME numbers - that agreement is the whole point of the exercise.
- The trick, restated: decision variables for a, b AND every predicted y, then ONE CONSTRAINT PER DATA POINT.
- FLAG THE DOMAIN: Reals, not NonNegativeReals. A slope or intercept can be negative. This trips people every time.
- Notice every loop is range(n_points) - never range(10). That is what makes it a template instead of a one-off.
- CLOSE: "scipy did it in one line. You just did it in twenty - and now you know what that one line was doing."
-->

## The same model in Pyomo

**The trick, restated:** we make decision variables for $a$, $b$ **and every predicted value** $\hat{y}_i$ - then add **one constraint per data point** tying each prediction to the model:

$$\hat{y}_i = a + b x_i$$

and minimize $\sum_i (\hat{y}_i - z_i)^2$.

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

# decision variables - note Reals, NOT NonNegativeReals (a slope can be negative!)
model.a = Var(domain=Reals, initialize=2000)
model.b = Var(domain=Reals, initialize=50)
model.y = Var(range(n_points), domain=Reals, initialize=0)

# one constraint per data point - the loop is driven by the CSV, not by a hard-coded 10
model.constraints = ConstraintList()
for i in range(n_points):
  model.constraints.add(model.y[i] == model.a + model.b*X[i])

# minimize the sum of squared errors
obj_expr = 0
for i in range(n_points):
  obj_expr += (model.y[i] - Z[i])**2
model.error = Objective(expr=obj_expr, sense=minimize)

SolverFactory('ipopt', executable='/content/bin/ipopt').solve(model)

print('a = %.4f' % model.a())
print('b = %.4f' % model.b())
print('sum of squared errors = %.2f' % model.error())

**Same answer as `scipy`** - $a \approx 4435.08$, $b \approx 47.07$. Which is exactly what you want to see: the one-line convenience function and the twenty-line optimization model agree, so now you know what that one line was doing.

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
Pharmacy from CSV - part 3: absolute error, and why it needs a reformulation

- Squared error punishes big misses hard. Absolute error treats a $500 miss as a $500 miss either way.
- You cannot type abs() - the solver follows the SLOPE, and |e| is a sharp V with no slope at the bottom. And the bottom is exactly where it is trying to get to.
- The fix: a new variable d per point, forced to be at least the error AND at least minus the error.
- The line that unsticks it: "one of those is always positive and the other negative, so bigger than both means bigger than the positive one - and that IS the absolute value."
- Then: minimizing is what pulls d down onto |e|. Do NOT say d equals |e| - you never wrote that.
- Answer comes out a = 4890, b = 39.33 - a DIFFERENT line from least squares. Show both on one plot and ask which they would defend.
-->

## Now with absolute error

Squared error punishes big misses hard. **Absolute error** treats a \$500 miss as a \$500 miss no matter which direction it went - often closer to what a business actually cares about.

But you **can't just type `abs()`**. See the 🔷 explainer in `Pharmacy - Complete_guided.ipynb` for the full story; the short version is that $|e|$ is a sharp **V** with no slope at the bottom, and the bottom is exactly where the solver is trying to land.

**The fix:** one new variable $d_i \ge 0$ per data point, pinned by **two** constraints - $d_i \ge e_i$ and $d_i \ge -e_i$. Bigger than both means bigger than the positive one, which *is* the absolute value. Then **minimizing** presses each $d_i$ down onto it.

In [ ]:
model2 = ConcreteModel()

model2.a = Var(domain=Reals, initialize=2000)
model2.b = Var(domain=Reals, initialize=50)
model2.y = Var(range(n_points), domain=Reals, initialize=0)

# one variable per data point to stand in for |error|
model2.d = Var(range(n_points), domain=NonNegativeReals)

model2.constraints = ConstraintList()
for i in range(n_points):
  model2.constraints.add(model2.y[i] == model2.a + model2.b*X[i])
  model2.constraints.add(model2.d[i] >=  (model2.y[i] - Z[i]))   # d >= error
  model2.constraints.add(model2.d[i] >= -(model2.y[i] - Z[i]))   # d >= -error

# minimize the sum of the d's - THIS is what pulls each d down onto |error|
obj_expr = 0
for i in range(n_points):
  obj_expr += model2.d[i]
model2.error = Objective(expr=obj_expr, sense=minimize)

SolverFactory('ipopt', executable='/content/bin/ipopt').solve(model2)

print('a = %.4f' % model2.a())
print('b = %.4f' % model2.b())
print('sum of absolute errors = %.2f' % model2.error())

In [ ]:
# both lines on one plot - they are NOT the same line
xs = np.linspace(min(X), max(X), 100)

plt.figure(figsize=(7.5,5))
plt.plot(X, Z, 'o', label='data')
plt.plot(xs, model.a()  + model.b()*xs,  lw=2, label='squared error')
plt.plot(xs, model2.a() + model2.b()*xs, lw=2, ls='--', label='absolute error')
plt.xlabel('pharmacy hours open'); plt.ylabel('revenue ($)')
plt.title('Two definitions of "best fit"'); plt.legend(); plt.grid(alpha=.3)
plt.show()

**Two different lines from the same data.** Squared error chases the big misses; absolute error shrugs at them. Neither is *wrong* - **you chose what counts as a big mistake**, and the solver did what you asked.

> **Remember:** the trick above only works because we're **minimizing**. If you maximized a sum of $d_i$, they'd run off to infinity - those constraints hold them up from below, never down from above.

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
Pharmacy from CSV - part 4: let the curve bend

- Back to the shape we noticed at the start - revenue does not climb in a straight line forever.
- Fit a POWER model and let the solver choose the exponent instead of you.
- The punchline number: b comes out around 0.48 - basically a SQUARE ROOT. The model discovered the shape on its own.
- Business reading: diminishing returns. The 100th hour is worth less than the 40th - which is the same idea as the shadow price decay in M3.1.
- CLOSE: "same CSV, same five Pyomo steps, three different models. Adding data means editing a file, not the notebook."
-->

## Let the curve bend: a power model

Back to the shape we noticed at the start. Instead of a straight line, fit

$$\hat{y}_i = a \cdot x_i^{\,b}$$

and **let the solver find the exponent $b$** rather than assuming one.

In [ ]:
model3 = ConcreteModel()

model3.a = Var(domain=Reals, initialize=1000)
model3.b = Var(domain=Reals, initialize=0.5, bounds=(0.01, 3))   # keep the search sensible
model3.y = Var(range(n_points), domain=Reals, initialize=1000)

model3.constraints = ConstraintList()
for i in range(n_points):
  model3.constraints.add(model3.y[i] == model3.a * X[i]**model3.b)

obj_expr = 0
for i in range(n_points):
  obj_expr += (model3.y[i] - Z[i])**2
model3.error = Objective(expr=obj_expr, sense=minimize)

SolverFactory('ipopt', executable='/content/bin/ipopt').solve(model3)

print('a = %.4f' % model3.a())
print('b = %.4f' % model3.b())
print('sum of squared errors = %.2f' % model3.error())

In [ ]:
plt.figure(figsize=(7.5,5))
plt.plot(X, Z, 'o', label='data')
plt.plot(xs, model.a() + model.b()*xs, lw=2, label='linear (squared error)')
plt.plot(xs, model3.a() * xs**model3.b(), lw=2, color='green', label='power model')
plt.xlabel('pharmacy hours open'); plt.ylabel('revenue ($)')
plt.title('Straight line vs. bending curve'); plt.legend(); plt.grid(alpha=.3)
plt.show()

**Look at the exponent.** It comes out around **0.48** - which is essentially a **square root**. Nobody told the model that; it found the shape from the data.

And it has a business reading: **diminishing returns.** The 100th hour of pharmacy time is worth less than the 40th. That's the same idea as the **shadow price decaying** in M3.1 - *diamonds into sand* - showing up in a completely different model.

> **Caution:** the power model fits better, but with **ten data points** you should be suspicious of any model that fits *too* well. Fitting the noise is easy; fitting the signal is the job.

## On your own

1. **Add rows to the CSV** (or point `url` at your own file) and re-run everything. Nothing in the model should need editing - that's the test of whether it's really soft-coded.
2. Fit the power model with **absolute error** instead of squared error. Does the exponent move?
3. Add a **polynomial** term ($a + bx + cx^2$) and compare all three fits on one plot.
4. Which model would you actually take to the pharmacy chain, and what would you say about the ten data points?

## Bottom line

**The model didn't change - only where the data came from.**

- Data lives in a **CSV**, read in one line, and every loop runs over `n_points`. Add stores, change nothing.
- **Regression is optimization**: decision variables for the coefficients *and* the predictions, one constraint per data point, minimize the error.
- **You pick what 'error' means.** Squared and absolute give genuinely different lines.
- **`abs()` needs a reformulation**, and it works only because you're minimizing.
- **Letting the curve bend** found a square-root shape and, with it, diminishing returns.

This is the shape of every fitting problem you'll meet from here - and now it reads its data like a real project would.